In [1]:
from globals import *
from objects import *
import hydra
import sys
import numpy as np
import time
import itertools

sys.path.insert(0, '../grid_gen')

from omegaconf import DictConfig
from hydra.utils import to_absolute_path

from env.load_map import load_map
from persona.cognitive.perceive import get_obs, print_visible
from persona.cognitive.plan import get_path, get_next_plan_and_waypoint, parse_current, parse_next, get_goal_coordinates, path_to_minigrid_actions

import json
from minigrid.minigrid_env import MiniGridEnv
from minigrid.core.grid import Grid
from minigrid.core.mission import MissionSpace
from env.world_object import make_tile         
from env.constants import SEM_TO_ID            

def heading_deg_to_dir(deg: int) -> int:
    return int(((deg % 360) + 45) // 90) % 4

def _in_bounds(x, y, W, H):
    return 0 <= x < W and 0 <= y < H

class DynamicMap(MiniGridEnv):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 1}

    def __init__(self,
                 json_path,
                 agent_view_size=7,
                 render_mode=None,
                 max_steps=10_000,
                 fire_start_loc = (0,0),
                 fire_spread_rate = 0.05,
                 traffic_disappear_mode=True,
                 traffic_disappear_prob=0.1,
                 traffic_concentration_pct=1.0,
                 traffic_concentration_align=0.5,
                 traffic_span=None):
        self.json_path = json_path
        elements, (W, H), agents = load_map(self.json_path)
        self._elements = elements
        self._W, self._H = W, H

        with open(self.json_path, "r") as f:
            raw = json.load(f)
        self._agent_spec = None
        if agents:
            a0 = agents[0]
            self._agent_spec = {
                "x": int(a0.get("start", {}).get("x", 0)),
                "y": int(a0.get("start", {}).get("y", 0)),
                "dir": heading_deg_to_dir(int(a0.get("heading_deg", a0.get("heading", 0)))),
            }
        elif "agent_start" in raw:
            sx, sy = tuple(raw["agent_start"].values())
            self._agent_spec = {"x": int(sx), "y": int(sy), "dir": 0}

        mission_space = MissionSpace(mission_func=lambda: "Human evacuation")
        super().__init__(
            width=W, height=H, max_steps=max_steps,
            mission_space=mission_space,
            agent_view_size=agent_view_size,
            see_through_walls=False,
            render_mode=render_mode,
        )

        self.traffic_objects = []
        self.traffic_lane_configs = []
        self.traffic_disappear_mode = traffic_disappear_mode
        self.traffic_disappear_prob = traffic_disappear_prob 
        self.traffic_concentration_pct = traffic_concentration_pct 
        self.traffic_concentration_align = traffic_concentration_align
        self.target_concentration_x = traffic_concentration_align
        self.traffic_span = traffic_span

        self.fire_start_loc = fire_start_loc
        self.fire_locations = []
        self.smoke_locations = []
        self.fire_spread_rate = fire_spread_rate

        self._last_view_size = agent_view_size
        self.goal_pos = None

    def _gen_grid(self, width, height):
        self.grid = Grid(self._W, self._H)
        for obj_type, rgba, color_name, sem_name, (x, y) in self._elements:
            tile = make_tile(sem_name=sem_name, rgba=rgba, name=None)
            self.grid.set(x, y, tile)

        # perimeter for now - bugs without it
        wall_rgba = (110, 130, 140)
        for x in range(width):
            self.grid.set(x, 0,              make_tile("block", wall_rgba))
            self.grid.set(x, height - 1,     make_tile("block", wall_rgba))
        for y in range(height):
            self.grid.set(0,          y,     make_tile("block", wall_rgba))
            self.grid.set(width - 1,  y,     make_tile("block", wall_rgba))

        self._place_agent_from_json()
        # start_x = self.width // 2
        # start_y = self.height // 2
        fire_start_x, fire_start_y = self.fire_start_loc
        self.grid.set(fire_start_x, fire_start_y, FireObstacle())
        self.fire_locations = [(fire_start_x, fire_start_y)]

        for i in range(self._H):
            self._add_traffic(row=i, initial_dir=1)
            self._add_traffic(row=i, initial_dir=1)
            self._add_traffic(row=i, initial_dir=1)

        # goal_id = SEM_TO_ID.get('home_A')
        # for i, j in itertools.product(range(width), range(height)):
        #     cell = self.grid.get(i, j)
        #     print(cell.sem_id)
        #     if cell and hasattr(cell, "sem_id") and cell.sem_id == goal_id:
        #     # if isinstance(cell, Goal):
        #         self.goal_pos = (i, j)
        #         print(self.goal_pos)
        #         break

    def _add_traffic(self, row, initial_dir):
        lane_config = {
            'row': row,
            'color': 'red',
            'initial_dir': initial_dir,
            'lane_start_x': 1,
            'lane_end_x': self.width - 2,
        }
        self.traffic_lane_configs.append(lane_config)

    def _get_concentration_zone(self, lane_start_x, lane_end_x):
        if self.target_concentration_x is not None and self.traffic_span is not None:
            center_x = self.target_concentration_x
            span = self.concentration_span
            half_span = span // 2
            
            zone_start_x = center_x - half_span
            zone_end_x = center_x + (span - 1) - half_span
            
            zone_start_x = max(lane_start_x, zone_start_x)
            zone_end_x = min(lane_end_x, zone_end_x)

            if zone_start_x > zone_end_x:
                zone_start_x = min(lane_end_x, center_x)
                zone_end_x = zone_start_x
            
            return zone_start_x, zone_end_x
        
        else:            
            lane_length = lane_end_x - lane_start_x + 1
            concentrated_width = max(1, int(lane_length * self.traffic_concentration_pct))
            remaining_width = lane_length - concentrated_width
            offset = int(remaining_width * self.traffic_concentration_align)
            
            zone_start_x = lane_start_x + offset
            zone_end_x = zone_start_x + concentrated_width - 1
            
            zone_end_x = min(zone_end_x, lane_end_x)
            
            return zone_start_x, zone_end_x
        
    def _update_traffic_state(self):
        if self.traffic_disappear_mode:
            active_traffic_objects = self.traffic_objects
            self.traffic_objects = []
            traffic_configs_for_respawn = []
            for traffic in active_traffic_objects:
                if np.random.rand() < self.traffic_disappear_prob:
                    self.grid.set(*traffic.cur_pos, None)
                    config = next((c for c in self.traffic_lane_configs if c['row'] == traffic.lane_row), None)
                    if config:
                        traffic_configs_for_respawn.append(config)
                else:
                    self.traffic_objects.append(traffic)
            active_rows = {t.lane_row for t in self.traffic_objects}
            all_respawn_configs = traffic_configs_for_respawn
            for config in self.traffic_lane_configs:
                if config['row'] not in active_rows:
                    all_respawn_configs.append(config)
            for config in all_respawn_configs:
                if np.random.rand() < self.traffic_disappear_prob:
                    lane_row = config['row']
                    lane_start_x = config['lane_start_x']
                    lane_end_x = config['lane_end_x']
                    zone_start_x, zone_end_x = self._get_concentration_zone(
                        lane_start_x, lane_end_x
                    )
                    random_col = self.np_random.integers(zone_start_x, zone_end_x + 1)
                    random_pos = (random_col, lane_row)
                    if self.grid.get(*random_pos) is None and random_pos != self.agent_pos:
                        new_traffic = TrafficObstacle(config['color'], config['initial_dir'])
                        new_traffic.lane_row = lane_row
                        new_traffic.lane_start_x = lane_start_x
                        new_traffic.lane_end_x = lane_end_x
                        new_traffic.cur_pos = random_pos

                        self.grid.set(random_pos[0], random_pos[1], new_traffic)
                        self.traffic_objects.append(new_traffic)
        
        else:
            for traffic in self.traffic_objects:
                old_pos = traffic.cur_pos
                self.grid.set(old_pos[0], old_pos[1], None)
                new_x = old_pos[0] + traffic.current_dir
                new_y = old_pos[1]
                if new_x > traffic.lane_end_x or new_x < traffic.lane_start_x:
                    traffic.current_dir *= -1
                    new_x = old_pos[0] + traffic.current_dir # Re-calculate new x
                traffic.cur_pos = (new_x, new_y)
                self.grid.set(new_x, new_y, traffic)
        
    def _calculate_min_fire_distance(self):
        if not self.fire_locations:
            return float('inf')
        
        min_dist = float('inf')
        ax, ay = self.agent_pos
        
        for fx, fy in self.fire_locations:
            dist = abs(ax - fx) + abs(ay - fy)
            if dist < min_dist:
                min_dist = dist
                
        return min_dist
    
    def _get_dynamic_view_size(self, min_dist):
        if min_dist >= 4:
            return 7
        elif min_dist == 3:
            return 5
        else: # min_dist <= 2
            return 3
        
    def _nearest_free(self, x0, y0, max_radius=4):
        W, H = self.width, self.height
        for r in range(max_radius + 1):
            for dx in range(-r, r + 1):
                for dy in range(-r, r + 1):
                    x, y = x0 + dx, y0 + dy
                    if not _in_bounds(x, y, W, H):
                        continue
                    cell = self.grid.get(x, y)
                    if cell is None or (hasattr(cell, "can_overlap") and cell.can_overlap()):
                        return (x, y)
        return None
    
    def _spread_fire(self, initial_placement=False):
        for loc in self.smoke_locations:
            if not isinstance(self.grid.get(*loc), FireObstacle):
                self.grid.set(*loc, None)
        self.smoke_locations.clear()
        
        new_fires = []
        if not initial_placement:
            current_fire_locs = list(self.fire_locations) 
            
            spread_deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]
            
            for fx, fy in current_fire_locs:
                for dx, dy in spread_deltas:
                    nx, ny = fx + dx, fy + dy
                    
                    in_bounds = (1 <= nx < self.width - 1 and 
                                 1 <= ny < self.height - 1)

                    if (in_bounds and 
                        self.grid.get(nx, ny) is None and
                        (nx, ny) != tuple(self.agent_pos) and
                        (nx, ny) != self.goal_pos): 
                        
                        if self.np_random.uniform() < self.fire_spread_rate:
                            self.grid.set(nx, ny, FireObstacle())
                            new_fires.append((nx, ny))
            
            self.fire_locations.extend(new_fires)
        
        potential_smoke_locs = set() 
        neighbor_deltas = [
            (-1, 0), (1, 0), (0, -1), (0, 1), 
            (-1, -1), (1, 1), (-1, 1), (1, -1)
        ]

        for fx, fy in self.fire_locations:
            for dx, dy in neighbor_deltas:
                sx, sy = fx + dx, fy + dy
                
                if (1 <= sx < self.width - 1 and 
                    1 <= sy < self.height - 1 and 
                    self.grid.get(sx, sy) is None and 
                    (sx, sy) != tuple(self.agent_pos) and 
                    (sx, sy) != self.goal_pos):
                    
                    potential_smoke_locs.add((sx, sy))

        for sx, sy in potential_smoke_locs:
            if self.grid.get(sx, sy) is None:
                self.grid.set(sx, sy, SmokeObstacle())
                self.smoke_locations.append((sx, sy))

    def gen_obs(self):
        min_dist = self._calculate_min_fire_distance()
        new_view_size = self._get_dynamic_view_size(min_dist)
        
        self.agent_view_size = new_view_size
        self._last_view_size = new_view_size

        obs = super().gen_obs()
        return obs

    def _place_agent_from_json(self):
        """
        Place a single controllable agent from JSON.
        If blocked, fallback to place_agent(); keep JSON heading.
        """
        if not self._agent_spec:
            self.place_agent()
            return

        ax, ay, d = self._agent_spec["x"], self._agent_spec["y"], self._agent_spec["dir"]
        cell = self.grid.get(ax, ay)
        if cell is None or (hasattr(cell, "can_overlap") and cell.can_overlap()):
            self.agent_pos = (ax, ay)
            self.agent_dir = d
        else:
            spot = self._nearest_free(ax, ay)
            if spot:
                self.agent_pos = spot
                self.agent_dir = d
            else:
                self.place_agent()
                self.agent_dir = d 

    def step(self, action):
        obs, reward, terminated, truncated, info = super().step(action)
        info['view_size'] = self._last_view_size
        agent_pos_after_action = self.agent_pos
        traffic_hit_by_agent = self.grid.get(*agent_pos_after_action)
        if traffic_hit_by_agent and traffic_hit_by_agent.type == 'ball':
             reward = -1.0
             terminated = True
             return obs, reward, terminated, truncated, info
        self._update_traffic_state()
        traffic_hit_agent = self.grid.get(*agent_pos_after_action)
        if traffic_hit_agent and traffic_hit_agent.type == 'ball':
             reward = -1.0
             terminated = True
             obs = self.gen_obs()
             return obs, reward, terminated, truncated, info

        if isinstance(self.grid.get(*self.agent_pos), FireObstacle):
            print("AGENT TOUCHED FIRE! 🔥")
            reward = -1.0
            terminated = True 

        if not terminated and not truncated:
            self._spread_fire()

        return obs, reward, terminated, truncated, info
        # return super().step(action)
    
    def reset(self, *, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options) 
        info['view_size'] = self._last_view_size
        return obs, info

/Users/waldburger/opt/anaconda3/envs/eLW085/lib/python3.9/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
# import heapq

# def smart_action_choice(goal_pos, env):
#     ax, ay = env.agent_pos
#     gx, gy = goal_pos

#     W, H = env._W, env._H

#     # Directions: E, S, W, N  (MiniGrid convention)
#     DIRS = [(1, 0), (0, 1), (-1, 0), (0, -1)]

#     FORWARD = env.actions.forward
#     TURN_LEFT = env.actions.left
#     TURN_RIGHT = env.actions.right

#     # ----------------------------
#     # Helpers
#     # ----------------------------
#     def in_bounds(x, y):
#         return 0 <= x < W and 0 <= y < H

#     def is_blocking(cell):
#         """
#         Match your original semantics:
#         - TrafficObstacle and FireObstacle are always blocked
#         - Anything that cannot be overlapped is blocked
#         """
#         if cell is None:
#             return False
#         # These names already exist in your original code
#         if isinstance(cell, (TrafficObstacle, FireObstacle)):
#             return True
#         return not cell.can_overlap()

#     def passable(x, y):
#         cell = env.grid.get(x, y)
#         return not is_blocking(cell)

#     def manhattan(x, y):
#         return abs(x - gx) + abs(y - gy)

#     def danger_penalty(x, y):
#         """
#         Extra cost for being adjacent to dangerous obstacles.
#         This makes the planner prefer wider paths around them.
#         """
#         penalty = 0
#         for dx in (-1, 0, 1):
#             for dy in (-1, 0, 1):
#                 if dx == 0 and dy == 0:
#                     continue
#                 nx, ny = x + dx, y + dy
#                 if not in_bounds(nx, ny):
#                     continue
#                 cell = env.grid.get(nx, ny)
#                 if isinstance(cell, (TrafficObstacle, FireObstacle)):
#                     penalty += 5  # tune as needed
#         return penalty

#     # ----------------------------
#     # A* PATHFINDING
#     # ----------------------------
#     def heuristic(x, y):
#         return manhattan(x, y)

#     start = (ax, ay)
#     goal = (gx, gy)

#     # If the goal itself is blocked, aim for the closest free cell near it.
#     if not passable(gx, gy):
#         # Search a small radius around the goal for a free cell
#         candidates = []
#         for dx in range(-2, 3):
#             for dy in range(-2, 3):
#                 tx, ty = gx + dx, gy + dy
#                 if in_bounds(tx, ty) and passable(tx, ty):
#                     candidates.append((manhattan(tx, ty), (tx, ty)))
#         if candidates:
#             _, (gx, gy) = min(candidates, key=lambda x: x[0])
#             goal = (gx, gy)
#         else:
#             # No reachable free cell near goal; just do something non-stuck
#             return TURN_LEFT

#     frontier = []
#     heapq.heappush(frontier, (0, start))
#     came_from = {start: None}
#     cost_so_far = {start: 0}

#     found_path = False

#     while frontier:
#         _, (x, y) = heapq.heappop(frontier)

#         if (x, y) == goal:
#             found_path = True
#             break

#         for dx, dy in DIRS:
#             nx, ny = x + dx, y + dy
#             if not in_bounds(nx, ny):
#                 continue
#             if not passable(nx, ny):
#                 continue

#             extra_cost = 1 + danger_penalty(nx, ny)
#             new_cost = cost_so_far[(x, y)] + extra_cost

#             if (nx, ny) not in cost_so_far or new_cost < cost_so_far[(nx, ny)]:
#                 cost_so_far[(nx, ny)] = new_cost
#                 priority = new_cost + heuristic(nx, ny)
#                 heapq.heappush(frontier, (priority, (nx, ny)))
#                 came_from[(nx, ny)] = (x, y)

#     # ----------------------------
#     # No path found → don't freeze
#     # ----------------------------
#     if not found_path:
#         # Simple non-stuck fallback: just turn left
#         return TURN_LEFT

#     # ----------------------------
#     # Reconstruct path: get next step
#     # ----------------------------
#     cx, cy = goal
#     while came_from[(cx, cy)] != start:
#         cx, cy = came_from[(cx, cy)]
#     next_step = (cx, cy)

#     # ----------------------------
#     # Convert next step → action
#     # ----------------------------
#     dx = next_step[0] - ax
#     dy = next_step[1] - ay

#     try:
#         target_dir = DIRS.index((dx, dy))
#     except ValueError:
#         # Shouldn't happen, but be safe
#         return TURN_LEFT

#     curr_dir = env.agent_dir

#     if curr_dir == target_dir:
#         # Before moving, double-check we're not stepping into a dangerous obstacle
#         nx, ny = ax + DIRS[curr_dir][0], ay + DIRS[curr_dir][1]
#         if in_bounds(nx, ny):
#             cell = env.grid.get(nx, ny)
#             if isinstance(cell, (TrafficObstacle, FireObstacle, SmokeObstacle)) or is_blocking(cell):
#                 # Path got invalidated by a moving obstacle; reorient instead
#                 return TURN_LEFT
#         return FORWARD

#     # Turn the shortest way toward the target direction
#     # (curr_dir - target_dir) % 4 == 1 means left turn is shorter
#     if (curr_dir - target_dir) % 4 == 1:
#         return TURN_LEFT
#     else:
#         return TURN_RIGHT

In [3]:
import heapq

# ============= GPT suggestion for action choice =============

def smart_action_choice(goal_pos, env):
    ax, ay = env.agent_pos
    gx, gy = goal_pos

    W, H = env._W, env._H

    # Directions: E, S, W, N  (MiniGrid convention)
    DIRS = [(1, 0), (0, 1), (-1, 0), (0, -1)]

    FORWARD = env.actions.forward
    TURN_LEFT = env.actions.left
    TURN_RIGHT = env.actions.right

    # --------------------------------------
    # Helpers
    # --------------------------------------
    def in_bounds(x, y):
        return 0 <= x < W and 0 <= y < H

    def is_blocking(cell):
        if cell is None:
            return False
        if isinstance(cell, (TrafficObstacle, FireObstacle)):
            return True
        return not cell.can_overlap()

    def passable(x, y):
        cell = env.grid.get(x, y)
        return not is_blocking(cell)

    def manhattan(x, y):
        return abs(x - gx) + abs(y - gy)

    def danger_penalty(x, y):
        penalty = 0
        for dx in (-1, 0, 1):
            for dy in (-1, 0, 1):
                if dx == 0 and dy == 0:
                    continue
                nx, ny = x + dx, y + dy
                if not in_bounds(nx, ny):
                    continue
                cell = env.grid.get(nx, ny)
                if isinstance(cell, (TrafficObstacle, FireObstacle)):
                    penalty += 5
        return penalty

    # --------------------------------------
    # A* PATHFINDING
    # --------------------------------------
    def heuristic(x, y):
        return manhattan(x, y)

    start = (ax, ay)
    goal = (gx, gy)

    # If goal cell is blocked, pick nearest free alternative
    if not passable(gx, gy):
        candidates = []
        for dx in range(-2, 3):
            for dy in range(-2, 3):
                tx, ty = gx + dx, gy + dy
                if in_bounds(tx, ty) and passable(tx, ty):
                    candidates.append((manhattan(tx, ty), (tx, ty)))
        if candidates:
            _, goal = min(candidates, key=lambda x: x[0])
            gx, gy = goal
        else:
            return TURN_LEFT  # emergency fallback

    # Trivial: already at goal
    if start == goal:
        return TURN_LEFT

    frontier = []
    heapq.heappush(frontier, (0, start))
    came_from = {start: None}
    cost_so_far = {start: 0}

    found_path = False

    while frontier:
        _, (x, y) = heapq.heappop(frontier)

        if (x, y) == goal:
            found_path = True
            break

        for dx, dy in DIRS:
            nx, ny = x + dx, y + dy
            if not in_bounds(nx, ny):
                continue
            if not passable(nx, ny):
                continue

            new_cost = cost_so_far[(x, y)] + 1 + danger_penalty(nx, ny)

            if (nx, ny) not in cost_so_far or new_cost < cost_so_far[(nx, ny)]:
                cost_so_far[(nx, ny)] = new_cost
                priority = new_cost + heuristic(nx, ny)
                heapq.heappush(frontier, (priority, (nx, ny)))
                came_from[(nx, ny)] = (x, y)

    # --------------------------------------
    # FAILED SEARCH → escape turn
    # --------------------------------------
    if not found_path:
        return TURN_LEFT

    # --------------------------------------
    # Reconstruct path safely
    # --------------------------------------
    path = []
    cur = goal

    # IMPORTANT FIX:
    # Stop if cur is None or if we hit the start
    while cur is not None and cur != start:
        path.append(cur)
        cur = came_from.get(cur, None)

    if not path:
        return TURN_LEFT  # nothing to do

    next_step = path[-1]  # first move from start

    # --------------------------------------
    # Convert next step → action
    # --------------------------------------
    dx = next_step[0] - ax
    dy = next_step[1] - ay

    if (dx, dy) not in DIRS:
        return TURN_LEFT  # fail-safe

    target_dir = DIRS.index((dx, dy))
    curr_dir = env.agent_dir

    # If aligned → try forward
    if curr_dir == target_dir:
        nx, ny = ax + DIRS[curr_dir][0], ay + DIRS[curr_dir][1]
        if not in_bounds(nx, ny):
            return TURN_LEFT
        cell = env.grid.get(nx, ny)
        if isinstance(cell, (TrafficObstacle, FireObstacle, SmokeObstacle)) or is_blocking(cell):
            return TURN_LEFT  # avoid collision
        return FORWARD

    # Use shortest rotation
    if (curr_dir - target_dir) % 4 == 1:
        return TURN_LEFT
    else:
        return TURN_RIGHT

In [5]:
env = DynamicMap(
    json_path='../grid_gen/configs/maps/urbanWorld.json',
    agent_view_size=7,
    fire_start_loc=(20,9),
    render_mode='rgb_array'
)

rec_env = RecordVideo(
    env,
    video_folder="integration_test",
    name_prefix="integration_test",
    fps=5
)

env = rec_env

print("[Env] Resetting...")
obs, info = env.reset(seed=0)
frame = env.render()
done = False
steps = 0

get_next_plan_text = "go to work"
current_location_text = "home_A"
goal_pos = get_goal_coordinates('work', env.unwrapped.grid)

for i in range(MAX_STEPS):

    action = smart_action_choice(goal_pos, env.unwrapped)

    obs, reward, terminated, truncated, info = env.step(action)

    env.render()

    current_view_size = info.get('view_size', env.unwrapped.agent_view_size)
    print(
        f"Step: {i+1:02d}, Action: {action}, Reward: {reward:.4f}, "
        f"View Size: {current_view_size}x{current_view_size}, Terminated: {terminated}"
    )
    
    if terminated or truncated or env.unwrapped.agent_pos == goal_pos:
        print(f"Episode Ended at step {i+1}. Final Reward: {reward:.4f}")
        obs, info = env.reset()
        break

env.close()

[Env] Resetting...
Step: 01, Action: 2, Reward: 0.0000, View Size: 7x7, Terminated: False
Step: 02, Action: 2, Reward: 0.0000, View Size: 7x7, Terminated: False
Step: 03, Action: 2, Reward: 0.0000, View Size: 7x7, Terminated: False
Step: 04, Action: 0, Reward: 0.0000, View Size: 7x7, Terminated: False
Step: 05, Action: 2, Reward: 0.0000, View Size: 7x7, Terminated: False
Step: 06, Action: 2, Reward: 0.0000, View Size: 7x7, Terminated: False
Step: 07, Action: 2, Reward: 0.0000, View Size: 7x7, Terminated: False
Step: 08, Action: 1, Reward: 0.0000, View Size: 7x7, Terminated: False
Step: 09, Action: 2, Reward: 0.0000, View Size: 7x7, Terminated: False
Step: 10, Action: 2, Reward: 0.0000, View Size: 7x7, Terminated: False
Step: 11, Action: 2, Reward: 0.0000, View Size: 7x7, Terminated: False
Step: 12, Action: 0, Reward: 0.0000, View Size: 7x7, Terminated: False
Step: 13, Action: 2, Reward: 0.0000, View Size: 7x7, Terminated: False
Step: 14, Action: 2, Reward: 0.0000, View Size: 7x7, Termi